# Diabetes Preprocessing
---

## Load and filter data

In [40]:
import pandas as pd

RAW = "../data/raw"
df = pd.read_csv(f"{RAW}/diabetic_data.csv")

C:\Users\piedu\AppData\Local\Temp\ipykernel_24828\2929803960.py:4: DtypeWarning: Columns (10: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{RAW}/diabetic_data.csv")


### Fill in blanks with a name

most "missing" values are not really missing. 
- A blank A1C result means the test was never ordered. 
- A blank payer or specialty means it was not recorded. 

Those are categories, so we give them names right away instead of leaving them as NaN where later steps would silently drop or impute them.

In [41]:
pre = df.isna().sum()

# Blank lab results mean the test was not ordered
df[["A1Cresult", "max_glu_serum"]] = df[["A1Cresult", "max_glu_serum"]].fillna("NotTested")

# Blank text fields mean "not recorded", which is still a category
for col in ["race", "payer_code", "medical_specialty", "diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].fillna("Missing")

filled = df.isna().sum()
changed = pre[pre != filled]
print(changed)


race                  2273
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64


### Remove encounters that cannot be readmitted
A patient who died in hospital or went to hospice cannot come back within 30 days.

In [42]:
print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")
not_readmittable = [11, 13, 14, 19, 20, 21]

df = df[~df["discharge_disposition_id"].isin(not_readmittable)]
df = df.reset_index(drop=True)

print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")

101,766 encounters, 71,518 patients
99,343 encounters, 69,990 patients


removing small instance of encounters whose gender is recorded as "Unknown/Invalid"
miniscule and data entry error

In [43]:
print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")
df = df[df["gender"] != "Unknown/Invalid"]
df = df.reset_index(drop=True)
print(f"{len(df):,} encounters, {df['patient_nbr'].nunique():,} patients")

99,343 encounters, 69,990 patients
99,340 encounters, 69,987 patients


### Build Target
We collapse the three-level readmitted column (<30, >30, NO) into a binary target, called readmit_30, where 1 means the patient was readmitted within 30 days and 0 means everything else, then drop the original column so it can't leak into the features

In [44]:
df["readmit_30"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns="readmitted")

print(f"Positive rate: {df['readmit_30'].mean():.1%}")

Positive rate: 11.4%


### Drop columns that carry nothing
Three kinds of columns go now, before any encoding, so they never cost time or add noise
- weight is 97% blank
- encounter_id is a row number. We already have patient_nbr for splitting later
- Drug columns examide and citoglipton are "No" in every row. Seven more drugs have one or two non-"No" rows altogether

In [45]:
print(len(df.columns))
unhelpful_drugs = ["examide", "citoglipton", "acetohexamide", "tolbutamide", "troglitazone",
              "glipizide-metformin", "glimepiride-pioglitazone",
              "metformin-rosiglitazone", "metformin-pioglitazone"]

df = df.drop(columns=["weight", "encounter_id"] + unhelpful_drugs)

print(len(df.columns))

50
39


## Recoding
--- 

### Group the ID codes by meaning
Admission type, discharge disposition, and admission source are stored as integers, but they are labels. Two problems:
- Several codes mean NULL, "Not Available", or "Not Mapped", which are missing values hiding as numbers.
- Most codes are rare (under 100 rows), so one-hot encoding them would add dozens of noisy columns.

Rather than lumping every rare code into one "Other" bucket, each code is mapped to a larger group that shares a clinical meaning (e.g. every kind of referral, every kind of nursing facility), so the groups keep a real readmission trend. The maps live in `data/id_groups.py` and are checked in `01_eda`. The raw integer columns are dropped so they cannot be mistaken for numeric features.

In [46]:
import sys
sys.path.insert(0, "..")
from scripts.id_groups import apply_id_groups

# Adds admission_type, admission_source, discharge_disposition; drops the three _id columns
df = apply_id_groups(df)

print(f"admission_type: {len(df["admission_type"].unique())} unique values")
print(f"admission_source: {len(df["admission_source"].unique())} unique values")
print(f"discharge_disposition: {len(df["discharge_disposition"].unique())} unique values")

admission_type: 4 unique values
admission_source: 4 unique values
discharge_disposition: 7 unique values


### Make age a number
Age arrives as buckets instead of integers, such as "[70-80)". The buckets have a real order, so they're mapped 0 through 9.

In [47]:
age_order = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]

df["age"] = df["age"].map({label: i for i, label in enumerate(age_order)})

df["age"].sample(100)

56778    8
68963    8
89292    6
24653    5
22801    7
        ..
64015    5
79655    8
34646    8
98297    2
80174    7
Name: age, Length: 100, dtype: int64

### Group the diagnosis codes
There are 3 columns in the data called diag_1, diag_2, and diag_3 which hold 700 to 800 distinct ICD-9 codes. This is too many to one hot encode. the standard baseline is to collapse them into nine body-system groups. 

https://www.aapc.com/codes/icd9-codes-range/?srsltid=AYCX9vKHTY7z3aad8El3ga-c3QmBvappZLTKKWN03j4GnTvPpjAi2RPd

Codes starting with V or E are supplementary codes and go to "Other"

In [48]:
def icd9_group(code):

    if code == "Missing":
        return "Missing"
    
    if code[0] in "VE":
        return "Other"
    
    n = float(code)

    if 390 <= n <= 459 or n == 785: 
        return "Circulatory"
    
    if 460 <= n <= 519 or n == 786: 
        return "Respiratory"
    
    if 520 <= n <= 579 or n == 787: 
        return "Digestive"
    
    if int(n) == 250:               
        return "Diabetes"
    
    if 800 <= n <= 999:             
        return "Injury"
    
    if 710 <= n <= 739:             
        return "Musculoskeletal"
    
    if 580 <= n <= 629 or n == 788: 
        return "Genitourinary"
    
    if 140 <= n <= 239:             
        return "Neoplasms"
    
    return "Other"

diag_cols = ["diag_1", "diag_2", "diag_3"]
for col in diag_cols:
    df[col] = df[col].map(icd9_group)
    print(df[col].unique())


<StringArray>
[       'Diabetes',           'Other',       'Neoplasms',     'Circulatory',
     'Respiratory',          'Injury', 'Musculoskeletal',       'Digestive',
   'Genitourinary',         'Missing']
Length: 10, dtype: str
<StringArray>
[        'Missing',        'Diabetes',       'Neoplasms',     'Circulatory',
     'Respiratory',           'Other',          'Injury', 'Musculoskeletal',
   'Genitourinary',       'Digestive']
Length: 10, dtype: str
<StringArray>
[        'Missing',           'Other',     'Circulatory',        'Diabetes',
     'Respiratory',          'Injury',       'Neoplasms',   'Genitourinary',
 'Musculoskeletal',       'Digestive']
Length: 10, dtype: str


### Flagging lab tests as tested or not

EDA missingness table has A1Cresult of 84748 NaN (83.3%), so 16.7% of encounters had the test, since a blank result means it was never ordered. In the main finding of Strack et al. 2014, the paper that published this dataset they claimed that ordering the test matters, so we do the same. 

In [49]:
df["a1c_tested"]     = (df["A1Cresult"] != "NotTested").astype(int)
df["glucose_tested"] = (df["max_glu_serum"] != "NotTested").astype(int)

df["a1c_tested_and_changed"] = df["a1c_tested"] * (df["change"] == "Ch").astype(int)

print(df[["a1c_tested", "glucose_tested", "a1c_tested_and_changed"]].mean().map("{:.1%}".format))


a1c_tested                16.9%
glucose_tested             5.2%
a1c_tested_and_changed     9.5%
dtype: str


### Simplifying payer and specialty
Both columns are about half blank and have many rare levels. 
- medical_specialty has 73 levels. The 10 most common cover 91.8% of rows, the rest will be folded into "Other"
- payer_code stays as a category but also add Medicare and Medicaid flags, this is good for later to check whether 
  the model works equally well across payers



In [50]:
top_specialties = df["medical_specialty"].value_counts().index[:10]
df["medical_specialty"] = df["medical_specialty"].where(
    df["medical_specialty"].isin(top_specialties), "Other")

df["medicare"] = (df["payer_code"] == "MC").astype(int)
df["medicaid"] = (df["payer_code"] == "MD").astype(int)

### Reduce the medication columns
EDA produced clear evidence that counting how many drugs a patient is on told you nothing, because every group sat at 10 to 11%. The direction of change did matter, with insulin going Up or Down at 13 to 14% against 10% for no insulin.

build features from change direction, keep insulin as its own four-level column, and keep the existing change and diabetesMed columns

**NOTE**: individual drug columns stay for tree models, for logistic regression, drop all except insulin to keep the one-hot matrix small

In [51]:
drug_cols = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
             "glipizide", "glyburide", "pioglitazone", "rosiglitazone", "acarbose",
             "miglitol", "tolazamide", "insulin", "glyburide-metformin"]

df["n_drugs_up"]   = (df[drug_cols] == "Up").sum(axis=1)
df["n_drugs_down"] = (df[drug_cols] == "Down").sum(axis=1)
df["n_drugs_on"]   = (df[drug_cols] != "No").sum(axis=1)

print(t.rename(index={3: "3+"}).to_string(formatters={"readmit_30": "{:.1%}".format}), "\n")


                n readmit_30
n_drugs_on                  
0           22636       9.9%
1           46039      12.6%
2           21575      10.8%
3+           9090      10.4% 



### prior-visits total
Prior inpatient visits is a strong single predictor, emergency visits follow the same. A combined total of all prior visits is a standard composite in readmission scoring

In [52]:
df["total_prior_visits"] = (df["number_outpatient"]
                            + df["number_emergency"]
                            + df["number_inpatient"])

print(df[["number_inpatient", "total_prior_visits"]].corrwith(df["readmit_30"]).round(3))


number_inpatient      0.168
total_prior_visits    0.128
dtype: float64


## Save

In [53]:
from pathlib import Path

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / "diabetic_clean.csv", index=False)

print(df.shape)

(99340, 48)
